### Consolidação BRONZE -> PRATA: Telemetria EPS dos CubeSats

Este notebook tem como objetivo realizar a leitura dos 4 arquivos brutos (.xlsx) da camada bronze (um por satélite, com uma aba por sessão de coleta) e consolidar tudo em uma única tabela na camada prata, já com a tipagem correta e uma coluna de timestamp absoluta.

Fonte dos dados: On-Orbit EPS Dataset of 1U CubeSat Constellation (projeto BIRDS),
Mendeley Data, DOI 10.17632/8kp25ycf63.1, licença CC BY.

In [0]:
%pip install openpyxl

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# Depois que uma biblioteca é instalada usando %pip install, o interpretador python que já está rodando no cluster não "enxerga" ela automaticamente - ele precisa reiniciar para recarregar o ambiente com a lib disponível.
dbutils.library.restartPython()

In [0]:
import pandas as pd
# Pandas é utilizado para ler os arquivos .xlsx e fazer a limpeza inicial antes de virar tabela Spark. É mais prático para manipular planilha com múltiplas abas do que o leitor nativo do Spark.
from datetime import datetime, timedelta
# Será necessário para calcular, mais a frente, o timestamp absoluto (data da aba + segundos decorridos)

CATALOG = "cubesat_eps"
VOLUME_PATH = f"/Volumes/{CATALOG}/bronze/raw_files"
SATELLITES = ["NEPALISAT", "RAAVANA", "TSURU", "UGUISU"]

#### Padronização dos nomes das colunas

Os 4 arquivos possuem 37 colunas cada, no entanto, os cabeçalhos originais têm pequenas inconsistências de digitação entre satélites (por exemplo: "Time Stamp" e "Time stamp"). Por isso, ao invés de confiarmos no texto do cabeçalho, vamos renomear as colunas por **posição**, assumindo que a ordem das 37 colunas é a mesma em todos os arquivos (o que foi confirmado durante inspeção visual das tabelas).

A estrutura das colunas será: 1 timestamp relativo (segundos) + 5 faces do painel solar (+X, -X, +Y, +Z, -Z) x 3 grandezas (temperatura, tensão e corrente) x 2 versões (raw/hex do sensor e calibrado) + bateria (tensão, corrente, temperatura) x 2 versões.

In [0]:
COLUMN_NAMES = [
    "time_stamp_s",
    "tpy_raw", "tpy_c", "tpx_raw", "tpx_c", "tmz_raw", "tmz_c",
    "tmx_raw", "tmx_c", "tpz_raw", "tpz_c",
    "vpy_raw", "vpy_mv", "vpx_raw", "vpx_mv", "vmz_raw", "vmz_mv",
    "vmx_raw", "vmx_mv", "vpz_raw", "vpz_mv",
    "ipy_raw", "ipy_ma", "ipx_raw", "ipx_ma", "imz_raw", "imz_ma",
    "imx_raw", "imx_ma", "ipz_raw", "ipz_ma",
    "vbat_raw", "vbat_v", "ibat_raw", "ibat_ma", "tbat_raw", "tbat_c",
]

#### Interpretando Datas

Cada aba representa uma sessão de coleta em uma data específica (o nome da aba é a própria data). Formatos de datas encontrados nos 4 arquivos: "9 Nov 2020", "13 Feb 2021", "10 April 2021", "23 October 2020", "March 28" (sem ano informado, específico para o TSURU) e "January 24 2022". Na planilha TSURU, o ano só aparece explicitamente quando muda para 2022 - as abas de março a dezembro sem ano foram assumidas como 2021, por estarem em sequência cronológica contínua antes das abas de janeiro de 2022.

Ainda sobre a planilha TSURU, duas abas ("Test1 w batt", "Test2 wo batt") não seguem nenhum padrão de data - sendo referente a testes feitos em solo (pré-lançamento), e não dados em órbita. Foram marcadas com **is_ground_test = true** para serem excluídas ou tratadas à parte na análise.

In [0]:
# A função tenta interpretar primeiramente o nome das abas nos formatos "com ano".
# se não bate nenhum formato com ano, tenta os formatos "sem ano" e assume 2021 (caso específico TSURU).
# se nenhum dos formatos bateu (ex: "Test1 w batt"), retorna None para data e True para o segundo valor - sinalizando que a aba é um teste em solo.

def parse_sheet_date(sheet_name: str):
    name = sheet_name.strip()

    for fmt in ("%d %b %Y", "%d %B %Y", "%B %d %Y", "%b %d %Y"):
        try:
            return datetime.strptime(name, fmt).date(), False
        except:
            pass

    for fmt in ("%B %d", "%b %d"):
        try:
            dt = datetime.strptime(name, fmt)
            return dt.replace(year=2021).date(), False
        except ValueError:
            pass

    return None, True

#### Leitura e Consolidação de Todos Arquivos

Para cada satélite, abrimos o arquivo .xlsx e iteramos por todas as suas abas. Cada aba vira um dataframe, recebe as colunas de contexto (satélite, aba, data, se é teste em solo) e é empilhada num conjunto único. Algumas abas trazem linhas totalmente em branco no final (sobra de formatação do excel), que são descartadas utilizando time_stamp_s nulo como critério.

In [0]:
all_frames = []

for sat in SATELLITES:
    file_path = f"{VOLUME_PATH}/{sat}.xlsx"
    xls = pd.ExcelFile(file_path)

    for sheet_name in xls.sheet_names:
        df = pd.read_excel(xls, sheet_name=sheet_name, header=0)
        df.columns = COLUMN_NAMES[: len(df.columns)]

        # Algumas abas trazem linhas totalmente em branco no final (sobra de formatação do excel).
        # Descartamos elas usando time_stamp_s como referência, já que é a única coluna que nunca deve estar vazia.

        df = df.dropna(subset = ["time_stamp_s"]).reset_index(drop=True)

        sheet_date, is_test = parse_sheet_date(sheet_name)

        df["satellite_id"] = sat
        df["sheet_name"] = sheet_name
        df["sheet_date"] = sheet_date
        df["is_ground_test"] = is_test
        df["source_file"] = f"{sat}.xlsx"

        all_frames.append(df)

df_consolidado = pd.concat(all_frames, ignore_index=True)
print(f"Total de linhas consolidadas: {len(df_consolidado)}")
df_consolidado.head()

Total de linhas consolidadas: 32837


,time_stamp_s,tpy_raw,tpy_c,tpx_raw,tpx_c,tmz_raw,tmz_c,tmx_raw,tmx_c,tpz_raw,tpz_c,vpy_raw,vpy_mv,vpx_raw,vpx_mv,vmz_raw,vmz_mv,vmx_raw,vmx_mv,vpz_raw,vpz_mv,ipy_raw,ipy_ma,ipx_raw,ipx_ma,imz_raw,imz_ma,imx_raw,imx_ma,ipz_raw,ipz_ma,vbat_raw,vbat_v,ibat_raw,ibat_ma,tbat_raw,tbat_c,satellite_id,sheet_name,sheet_date,is_ground_test,source_file
0,0.0,06EA,-8.24,06FA,-10.02,071B,-13.68,06DB,-6.58,06C1,-3.69,0371,1344.63,033C,1263.74,0335,1253.05,0362,1321.73,0362,1321.73,00,0.00,00,0.0,01,4.66,02,9.32,01,4.66,9C,3.97,07BC,201.84,AF,7.96,NEPALISAT,9 Nov 2020,2020-11-09,False,NEPALISAT.xlsx
1,5.0,06EB,-8.35,06FC,-10.24,071C,-13.79,06DB,-6.58,06C1,-3.69,0371,1344.63,033C,1263.74,0335,1253.05,0362,1321.73,0362,1321.73,00,0.00,00,0.0,01,4.66,02,9.32,01,4.66,9C,3.97,07AE,242.84,AF,7.96,NEPALISAT,9 Nov 2020,2020-11-09,False,NEPALISAT.xlsx
2,10.0,06EC,-8.46,06FE,-10.46,071D,-13.90,06DC,-6.69,06C2,-3.80,0371,1344.63,033D,1265.26,0335,1253.05,0362,1321.73,0362,1321.73,00,0.00,00,0.0,01,4.66,02,9.32,01,4.66,9C,3.97,07AE,242.84,AF,7.96,NEPALISAT,9 Nov 2020,2020-11-09,False,NEPALISAT.xlsx
3,15.0,06ED,-8.57,06FE,-10.46,071D,-13.90,06DB,-6.58,06C3,-3.91,0371,1344.63,033C,1263.74,0334,1251.53,0361,1320.21,0362,1321.73,01,4.66,00,0.0,01,4.66,02,9.32,01,4.66,9C,3.97,07B0,236.98,AF,7.96,NEPALISAT,9 Nov 2020,2020-11-09,False,NEPALISAT.xlsx
4,20.0,06EF,-8.80,06FF,-10.57,071F,-14.12,06DC,-6.69,06C3,-3.91,0371,1344.63,033C,1263.74,0335,1253.05,0362,1321.73,0362,1321.73,01,4.66,00,0.0,01,4.66,02,9.32,01,4.66,9C,3.97,07AF,239.91,AF,7.96,NEPALISAT,9 Nov 2020,2020-11-09,False,NEPALISAT.xlsx


In [0]:
raw_cols = [c for c in df_consolidado.columns if c.endswith("_raw")]
df_consolidado[raw_cols] = df_consolidado[raw_cols].astype(str)

#### Convertendo Dados para Spark Dataframe

Nesta etapa é convertido o "df_consolidado" (pandas) em Spark Dataframe para podermos gravá-lo como tabela gerenciada no Unity Catalog (camada prata). As colunas "_raw" foram forçadas para o formato String na etapa anterior para evitar erro de inferência de tipo (ArrowTypeError) na conversão.

In [0]:
spark_df = spark.createDataFrame(df_consolidado)
spark_df.printSchema()

root
 |-- time_stamp_s: double (nullable = true)
 |-- tpy_raw: string (nullable = true)
 |-- tpy_c: double (nullable = true)
 |-- tpx_raw: string (nullable = true)
 |-- tpx_c: double (nullable = true)
 |-- tmz_raw: string (nullable = true)
 |-- tmz_c: double (nullable = true)
 |-- tmx_raw: string (nullable = true)
 |-- tmx_c: double (nullable = true)
 |-- tpz_raw: string (nullable = true)
 |-- tpz_c: double (nullable = true)
 |-- vpy_raw: string (nullable = true)
 |-- vpy_mv: double (nullable = true)
 |-- vpx_raw: string (nullable = true)
 |-- vpx_mv: double (nullable = true)
 |-- vmz_raw: string (nullable = true)
 |-- vmz_mv: double (nullable = true)
 |-- vmx_raw: string (nullable = true)
 |-- vmx_mv: double (nullable = true)
 |-- vpz_raw: string (nullable = true)
 |-- vpz_mv: double (nullable = true)
 |-- ipy_raw: string (nullable = true)
 |-- ipy_ma: double (nullable = true)
 |-- ipx_raw: string (nullable = true)
 |-- ipx_ma: double (nullable = true)
 |-- imz_raw: string (nullable =

#### Gravação da Tabela Prata

É gravado o "spark_df" consolidado como tabela gerenciada no Unity Catalog, schema "prata". Usamos *overwrite* porque estamos reconstruindo a tabela do zero nesta etapa.

In [0]:
spark_df.write.mode("overwrite").saveAsTable(f"{CATALOG}.prata.eps_telemetria")